# Your contact-center dashboard may be lying to you
## Three SQL traps in average handle time and first-contact resolution

A dashboard can contain perfectly valid SQL and still answer the wrong
question. This notebook uses **11 invented contacts** to show how an
innocent-looking denominator, an average of averages, or an incomplete
follow-up window changes the story.

**You will build:** a queue-level report with defensible AHT and FCR
definitions, then deliberately break the assumptions and inspect the result.

**Run all cells in order.** Python 3.10+ and SQLite 3.25+ are sufficient.
No downloads, package installation, GPU, network access, or private data.
All results describe this tiny synthetic fixture, not a client or employer.

Project direction: Neal Vazquez. Notebook, SQL and checks developed with
ChatGPT assistance. This notebook extends the
[public SQL case study](https://github.com/neal-vazquez/nealvazquez-public/tree/main/business/consulting-career/sql-and-analytics/examples/contact-center).


### 1. Define the question before the query

Which queue warrants investigation, and do we have enough evidence to
compare performance? AHT means average handle time. FCR means first-contact
resolution; here it is a **seven-day operational proxy**, not a survey measure.

| Quantity | Definition in this exercise |
| --- | --- |
| Contact | One logical contact after upstream deduplication and transfer consolidation |
| AHT | Known handle seconds / handled contacts with known duration |
| First contact | Earliest handled contact for a case across all observed history |
| Eligible FCR case | First contact in the reporting period, with a complete seven-day observation window |
| FCR numerator | Eligible cases marked resolved at first contact, with no handled repeat within seven days |
| Pending | Seven-day follow-up not complete; not a resolved or failed case yet |

Reporting: **September 1 through September 9, 2026, UTC**. Observations
stop strictly before September 16 at 00:00 UTC. Repeats after the reporting
period can still change the outcome of cases that started during it.


In [1]:
import json
import sqlite3
from datetime import datetime
from statistics import mean

assert sqlite3.sqlite_version_info >= (3, 25, 0), 'SQLite window functions required'
PARAMS = {
    'start_at': '2026-09-01 00:00:00',
    'end_at': '2026-09-10 00:00:00',
    'as_of': '2026-09-16 00:00:00',
}

def show_table(rows, columns=None):
    if not rows:
        print('(no rows)')
        return
    columns = columns or list(rows[0])
    print(' | '.join(columns))
    print(' | '.join('---' for _ in columns))
    for row in rows:
        print(' | '.join('NULL' if row.get(c) is None else str(row[c]) for c in columns))


In [2]:
SCHEMA_SQL = '-- Standalone synthetic exercise, unrelated to the website schema or client data.\n-- One row per logical contact, after source deduplication and transfer-leg consolidation.\nCREATE TABLE contacts (\n    contact_id TEXT PRIMARY KEY NOT NULL,\n    case_id TEXT NOT NULL,\n    queue TEXT NOT NULL,\n    started_at TEXT NOT NULL, -- canonical UTC YYYY-MM-DD HH:MM:SS\n    handled INTEGER NOT NULL CHECK (handled IN (0, 1)),\n    handle_seconds INTEGER CHECK (handle_seconds >= 0),\n    resolved INTEGER NOT NULL CHECK (resolved IN (0, 1)),\n    CHECK (handled = 1 OR (handle_seconds IS NULL AND resolved = 0))\n);\nCREATE INDEX contacts_case_time ON contacts(case_id, started_at, contact_id);\nCREATE INDEX contacts_time ON contacts(started_at);\n'


In [3]:
FIXTURE_SQL = "-- Invented contacts only. No customer identifiers, transcripts, or client records.\nINSERT INTO contacts VALUES\n('c01', 'issue-1', 'billing', '2026-09-01 09:00:00', 1, 300, 1),\n('c02', 'issue-2', 'billing', '2026-09-01 10:00:00', 1, 600, 1),\n('c03', 'issue-2', 'billing', '2026-09-03 10:00:00', 1, 300, 1),\n('c04', 'issue-3', 'technical', '2026-09-02 09:00:00', 1, 900, 0),\n('c05', 'issue-3', 'technical', '2026-09-04 09:00:00', 1, 600, 1),\n('c06', 'issue-4', 'billing', '2026-09-04 10:00:00', 0, NULL, 0),\n('c07', 'issue-5', 'technical', '2026-09-07 10:00:00', 1, NULL, 1),\n('c08', 'issue-6', 'billing', '2026-09-08 10:00:00', 1, 240, 1),\n('c09', 'issue-7', 'technical', '2026-09-08 11:00:00', 1, 480, 1),\n-- Follow-up outside the reporting period still disqualifies first-contact resolution.\n('c10', 'issue-7', 'technical', '2026-09-10 11:00:00', 1, 420, 1),\n-- Recent first contact lacks a complete seven-day follow-up window at the cutoff.\n('c11', 'issue-8', 'billing', '2026-09-09 11:00:00', 1, 180, 1);\n"


In [4]:
def connect():
    db = sqlite3.connect(':memory:')
    db.row_factory = sqlite3.Row
    db.executescript(SCHEMA_SQL)
    db.executescript(FIXTURE_SQL)
    return db

db = connect()
show_table([dict(r) for r in db.execute('SELECT * FROM contacts ORDER BY started_at, contact_id')])


contact_id | case_id | queue | started_at | handled | handle_seconds | resolved
--- | --- | --- | --- | --- | --- | ---
c01 | issue-1 | billing | 2026-09-01 09:00:00 | 1 | 300 | 1
c02 | issue-2 | billing | 2026-09-01 10:00:00 | 1 | 600 | 1
c04 | issue-3 | technical | 2026-09-02 09:00:00 | 1 | 900 | 0
c03 | issue-2 | billing | 2026-09-03 10:00:00 | 1 | 300 | 1
c05 | issue-3 | technical | 2026-09-04 09:00:00 | 1 | 600 | 1
c06 | issue-4 | billing | 2026-09-04 10:00:00 | 0 | NULL | 0
c07 | issue-5 | technical | 2026-09-07 10:00:00 | 1 | NULL | 1
c08 | issue-6 | billing | 2026-09-08 10:00:00 | 1 | 240 | 1
c09 | issue-7 | technical | 2026-09-08 11:00:00 | 1 | 480 | 1
c11 | issue-8 | billing | 2026-09-09 11:00:00 | 1 | 180 | 1
c10 | issue-7 | technical | 2026-09-10 11:00:00 | 1 | 420 | 1


### 2. Keep contact counts and case outcomes at their own grains

The query below ranks the complete observed case history **before**
selecting the reporting cohort. It aggregates contact metrics and case
outcomes separately, then joins the two summaries at queue grain.
`NOT EXISTS` detects repeats without multiplying rows and inflating totals.

A repeat exactly seven days later counts. A first contact whose seven-day
boundary equals the exclusive observation cutoff remains pending. Tied
contact timestamps use ID order for deterministic selection; the data
cannot establish an actual within-timestamp order.


In [5]:
METRICS_SQL = "-- Reporting period: [:start_at, :end_at); data observed strictly before :as_of.\n-- Seven-day FCR proxy: first handled contact marked resolved, no further handled\n-- contact on the same case within seven days, and complete follow-up observed.\n-- Equality at the maturity boundary stays pending because :as_of is exclusive.\nWITH observed AS (\n    SELECT * FROM contacts WHERE started_at < :as_of\n), volumes AS (\n    SELECT queue,\n           COUNT(*) AS offered_contacts,\n           SUM(handled) AS handled_contacts,\n           COUNT(*) - SUM(handled) AS abandoned_contacts,\n           SUM(CASE WHEN handled = 1 AND handle_seconds IS NOT NULL THEN 1 ELSE 0 END) AS timed_contacts,\n           SUM(CASE WHEN handled = 1 AND handle_seconds IS NULL THEN 1 ELSE 0 END) AS missing_handle_times,\n           SUM(CASE WHEN handled = 1 THEN COALESCE(handle_seconds, 0) ELSE 0 END) AS total_handle_seconds\n    FROM observed\n    WHERE started_at >= :start_at AND started_at < :end_at\n    GROUP BY queue\n), ranked AS (\n    -- Rank full observed history, before restricting the first-contact cohort.\n    SELECT *, ROW_NUMBER() OVER (\n        PARTITION BY case_id ORDER BY started_at, contact_id\n    ) AS contact_number\n    FROM observed WHERE handled = 1\n), first_contacts AS (\n    SELECT *, CASE WHEN datetime(started_at, '+7 days') < :as_of THEN 1 ELSE 0 END AS mature\n    FROM ranked\n    WHERE contact_number = 1 AND started_at >= :start_at AND started_at < :end_at\n), cohort AS (\n    SELECT f.queue, COUNT(*) AS first_contact_cases,\n           SUM(f.mature) AS eligible_cases,\n           SUM(CASE WHEN f.mature = 0 THEN 1 ELSE 0 END) AS pending_cases,\n           SUM(CASE WHEN f.mature = 1 AND f.resolved = 1 AND NOT EXISTS (\n               SELECT 1 FROM observed AS followup\n               WHERE followup.case_id = f.case_id AND followup.handled = 1\n                 AND followup.contact_id <> f.contact_id\n                 AND followup.started_at >= f.started_at\n                 AND followup.started_at <= datetime(f.started_at, '+7 days')\n           ) THEN 1 ELSE 0 END) AS fcr_cases\n    FROM first_contacts AS f\n    GROUP BY f.queue\n)\n-- Aggregate the two grains independently before joining to avoid fan-out.\nSELECT v.queue, v.offered_contacts, v.handled_contacts, v.abandoned_contacts,\n       v.timed_contacts, v.missing_handle_times, v.total_handle_seconds,\n       ROUND(1.0 * v.total_handle_seconds / NULLIF(v.timed_contacts, 0), 2) AS aht_seconds,\n       COALESCE(c.first_contact_cases, 0) AS first_contact_cases,\n       COALESCE(c.eligible_cases, 0) AS eligible_cases,\n       COALESCE(c.pending_cases, 0) AS pending_cases,\n       COALESCE(c.fcr_cases, 0) AS fcr_cases,\n       ROUND(100.0 * c.fcr_cases / NULLIF(c.eligible_cases, 0), 2) AS fcr_pct\nFROM volumes AS v\nLEFT JOIN cohort AS c USING (queue)\nORDER BY v.queue;\n"
print(METRICS_SQL)


-- Reporting period: [:start_at, :end_at); data observed strictly before :as_of.
-- Seven-day FCR proxy: first handled contact marked resolved, no further handled
-- contact on the same case within seven days, and complete follow-up observed.
-- Equality at the maturity boundary stays pending because :as_of is exclusive.
WITH observed AS (
    SELECT * FROM contacts WHERE started_at < :as_of
), volumes AS (
    SELECT queue,
           COUNT(*) AS offered_contacts,
           SUM(handled) AS handled_contacts,
           COUNT(*) - SUM(handled) AS abandoned_contacts,
           SUM(CASE WHEN handled = 1 AND handle_seconds IS NOT NULL THEN 1 ELSE 0 END) AS timed_contacts,
           SUM(CASE WHEN handled = 1 AND handle_seconds IS NULL THEN 1 ELSE 0 END) AS missing_handle_times,
           SUM(CASE WHEN handled = 1 THEN COALESCE(handle_seconds, 0) ELSE 0 END) AS total_handle_seconds
    FROM observed
    WHERE started_at >= :start_at AND started_at < :end_at
    GROUP BY queue
), ranked A

In [6]:
def validate_params(params):
    """Validate actual UTC calendar timestamps before lexical SQL comparison."""
    if not isinstance(params, dict):
        raise ValueError('Reporting parameters must be a dictionary.')
    for key in ('start_at', 'end_at', 'as_of'):
        value = params.get(key)
        if not isinstance(value, str):
            raise ValueError(f'{key} must be a canonical UTC YYYY-MM-DD HH:MM:SS string.')
        try:
            parsed = datetime.strptime(value, '%Y-%m-%d %H:%M:%S')
        except ValueError as exc:
            raise ValueError(f'{key} must be a valid UTC calendar timestamp.') from exc
        canonical = f'{parsed.year:04d}-{parsed.month:02d}-{parsed.day:02d} {parsed.hour:02d}:{parsed.minute:02d}:{parsed.second:02d}'
        if value != canonical:
            raise ValueError(f'{key} must use canonical UTC YYYY-MM-DD HH:MM:SS format.')
    if not params['start_at'] < params['end_at'] <= params['as_of']:
        raise ValueError('Require start_at < end_at <= as_of in canonical UTC format.')


In [7]:
def report(connection, params=None):
    params = PARAMS if params is None else params
    validate_params(params)
    return [dict(row) for row in connection.execute(METRICS_SQL, params)]

rows = report(db)
show_table(rows, ['queue', 'handled_contacts', 'timed_contacts',
                  'aht_seconds', 'eligible_cases', 'fcr_cases', 'fcr_pct', 'pending_cases'])
assert rows[0]['aht_seconds'] == 324
assert rows[1]['aht_seconds'] == 660
assert sum(r['offered_contacts'] for r in rows) == 10


queue | handled_contacts | timed_contacts | aht_seconds | eligible_cases | fcr_cases | fcr_pct | pending_cases
--- | --- | --- | --- | --- | --- | --- | ---
billing | 5 | 5 | 324.0 | 3 | 2 | 66.67 | 1
technical | 4 | 3 | 660.0 | 3 | 1 | 33.33 | 0


### Trap 1: Missing duration is not zero duration

Technical handled four contacts, but one duration is unknown. Dividing
known seconds by all four contacts silently treats the missing value as
zero. The observed-duration AHT uses three contacts and reports coverage.

Excluding the missing duration does **not** prove that 660 seconds is an
unbiased estimate of all technical contacts. The sensitivity calculation
below makes the missing-value assumption explicit.


In [8]:
technical = next(r for r in rows if r['queue'] == 'technical')
observed_aht = technical['total_handle_seconds'] / technical['timed_contacts']
zero_imputed_aht = technical['total_handle_seconds'] / technical['handled_contacts']
print(f'Observed-duration AHT: {observed_aht:.0f}s (3 of 4 contacts)')
print(f'Silent zero-imputation: {zero_imputed_aht:.0f}s')
print(f'Difference relative to observed-duration AHT: {100 * (zero_imputed_aht / observed_aht - 1):.1f}%')
sensitivity = [{'assumed_missing_seconds': missing,
                'all_four_contact_aht': (technical['total_handle_seconds'] + missing) / 4}
               for missing in [0, 300, 660, 1200]]
show_table(sensitivity)
assert (observed_aht, zero_imputed_aht) == (660, 495)


Observed-duration AHT: 660s (3 of 4 contacts)
Silent zero-imputation: 495s
Difference relative to observed-duration AHT: -25.0%
assumed_missing_seconds | all_four_contact_aht
--- | ---
0 | 495.0
300 | 570.0
660 | 660.0
1200 | 795.0


### Trap 2: Averaging queue averages changes the unit of analysis

Each queue does not contain the same number of timed contacts. If the
target is the average **contact**, reconstruct it from total seconds and
total timed contacts. An equal-weight queue mean answers a different question.


In [9]:
weighted_aht = sum(r['total_handle_seconds'] for r in rows) / sum(r['timed_contacts'] for r in rows)
unweighted_aht = mean(r['aht_seconds'] for r in rows)
print(f'Contact-weighted AHT: {weighted_aht:.0f}s')
print(f'Equal-weight queue mean: {unweighted_aht:.0f}s')
print(f'Overstatement for the contact-level question: {100 * (unweighted_aht / weighted_aht - 1):.2f}%')
assert (weighted_aht, unweighted_aht) == (450, 492)


Contact-weighted AHT: 450s
Equal-weight queue mean: 492s
Overstatement for the contact-level question: 9.33%


### Trap 3: An incomplete observation window is not a success

Billing has one recent first contact that is still pending. Calling it
resolved too early changes the denominator and numerator. Technical has
a repeat contact after the reporting period that must still affect FCR.

These are counterfactual edits to an in-memory copy of invented data.
They demonstrate measurement failure, not a causal business intervention.


In [10]:
billing = next(r for r in rows if r['queue'] == 'billing')
print(f"Billing mature-case FCR: {billing['fcr_cases']}/{billing['eligible_cases']} = {billing['fcr_pct']}%; pending: {billing['pending_cases']}")
print('Prematurely counting the pending case as resolved: 3/4 = 75%')

altered = connect()
altered.execute("DELETE FROM contacts WHERE contact_id = 'c10'")
incomplete = next(r for r in report(altered) if r['queue'] == 'technical')
altered.close()
print(f"Technical with complete follow-up: {technical['fcr_pct']}%")
print(f"Technical with the later repeat omitted: {incomplete['fcr_pct']}%")
assert incomplete['offered_contacts'] == technical['offered_contacts']
assert incomplete['fcr_cases'] == technical['fcr_cases'] + 1

boundary_rows = report(db, {**PARAMS, 'as_of': '2026-09-16 11:00:00'})
later_rows = report(db, {**PARAMS, 'as_of': '2026-09-16 11:00:01'})
assert boundary_rows[0]['pending_cases'] == 1
assert later_rows[0]['pending_cases'] == 0
print('Maturity boundary verified: equality remains pending; one second later it is eligible.')


Billing mature-case FCR: 2/3 = 66.67%; pending: 1
Prematurely counting the pending case as resolved: 3/4 = 75%
Technical with complete follow-up: 33.33%
Technical with the later repeat omitted: 66.67%
Maturity boundary verified: equality remains pending; one second later it is eligible.


### 3. Make the business conclusion as careful as the SQL

Technical has higher observed AHT and lower FCR in this fixture. That
warrants investigation, not a ranking of agent quality. There are only
three mature cases per queue, one missing duration, and no adjustment
for case complexity, routing, or channel mix.

| Tempting claim | What the evidence actually supports |
| --- | --- |
| Technical agents are slower | Observed timed contacts in Technical take longer in this invented sample |
| Cutting AHT improves service | No intervention or customer-experience outcome was measured |
| No repeat means resolved | Only within this observation window, linkage system, and explicit resolution proxy |
| A zero rate and missing rate are equivalent | An empty denominator yields NULL, which preserves uncertainty |

In a real analysis, investigate same-issue repeats and missing durations,
compare like cases, and evaluate resolution and customer experience
alongside handle time. Late ingestion needs a completeness watermark
and cohort restatement. This exercise does not model detailed transfer
legs, source revisions, business-hour calendars, or causal effects.


In [11]:
unknown_durations = connect()
unknown_durations.execute('UPDATE contacts SET handle_seconds = NULL WHERE handled = 1')
assert all(r['aht_seconds'] is None for r in report(unknown_durations))
unknown_durations.close()
empty = connect()
empty.execute('DELETE FROM contacts')
assert report(empty) == []
empty.close()
db.close()
print('All notebook assertions passed. Unknown is preserved as unknown.')


All notebook assertions passed. Unknown is preserved as unknown.


### Try a meaningful variation

Change one assumption at a time: move a repeat to exactly seven days,
place a first contact before the reporting window, or send a repeat to
another queue. Predict the effect before running it. Case outcomes belong
to the first handled contact's queue; contact volume belongs to the queue
that handled each contact.

Source, boundary tests, and reproducible build:
[nealvazquez-public](https://github.com/neal-vazquez/nealvazquez-public).
The notebook embeds the repository's synthetic fixture and SQL so it
works offline. Its source-sync test prevents the two versions drifting.
